# Chr22 QC + thinning + timing

First look at GRM-input variant QC, on a single chromosome before committing to a
genome-wide run: ACAF threshold callset, restricted to round 2b's ancestry-filtered
keep-list, standard hard-call QC (MAF, HWE, missingness, biallelic), then random
thinning tuned toward a ~1M-variant genome-wide target. Every plink2 call is timed —
chr22 is ~1.6% of the autosomal genome by length, so its QC time is the seed estimate
for sizing the genome-wide run and, later, the `GRM-pairs/grm_bin_sharded` shard
parallelization (`--parallel k n`) — not attempted here, this notebook stops at "how
long does QC take on one chromosome, and how many variants survive."

## Compute resource

Unlike the phenotype-stage notebooks (small BigQuery aggregates), this one reads a
real chromosome's worth of genotypes for every AoU participant before `--keep`
narrows it down, and plink2 QC/thinning is CPU + I/O bound. Workbench 2.0's default
(2 CPU / 13 GB) is too small — size up to something in the 8-16 vCPU / 32-64 GB range
and pass `--threads`/`--memory` to plink2 to actually use it. Still much lighter than
the eventual genome-wide GRM shard construction, which is squarely an AoU batch job,
not an interactive notebook.

## Setup

plink2: manual install, not conda — same as `01_ancestry_filtering`'s notebooks.
Bucket already mounted at `~/workspace/` by Verily Workbench, no `gcsfuse` step
needed.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/2.0/
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version

nproc
free -h

In [ ]:
import os

# %%bash cells don't inherit PATH changes from prior cells -- set it here instead
bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

N_THREADS = os.cpu_count()   # pass to plink2 --threads below; matches whatever
                              # compute profile this environment was actually sized to

## Paths

`ANCESTRY_BUCKET_DIR` / `ROUND2B_KEEP_PATH`: round 2b's ancestry-filtered keep-list
(`reverse_pca_aou.ipynb`'s AoU-fit ellipsoid output) — this is the sample set GRM
construction is actually for, so QC (MAF/HWE computed *within this cohort*, not the
full AoU population) restricts to it from the start via `--keep`.

`ACAF_PGEN_PATTERN`: same validated path `submit_pca_r1.ipynb` uses — native pgen
export, chromosome-split. Native pgen, not `plink_bed`: extraction against the
PLINK1 bed export over the network-mounted filesystem hangs indefinitely on this
scattered/large-scale access pattern; pgen doesn't (same lesson as
`submit_pca_r1.ipynb`).

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/01_ancestry_filtering"
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"
os.makedirs(BUCKET_DIR, exist_ok=True)

ROUND2B_KEEP_PATH = f"{ANCESTRY_BUCKET_DIR}/reverse_pca_aou/round2b_keep_ids_aou_fit_r2b_p99.9999.txt"
assert os.path.isfile(ROUND2B_KEEP_PATH), f"round 2b keep-list not found: {ROUND2B_KEEP_PATH!r}"

# AOU_ROOT: parent of the shared-env-pilot dir -- ACAF lives under a sibling
# directory, not under shared-env-pilot itself (same as submit_pca_r1.ipynb)
AOU_ROOT = os.path.dirname(WORKSPACE_BUCKET)
ACAF_PGEN_DIR = f"{AOU_ROOT}/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen"
assert os.path.isdir(ACAF_PGEN_DIR), f"ACAF_PGEN_DIR does not exist: {ACAF_PGEN_DIR!r}"
ACAF_PGEN_PATTERN = ACAF_PGEN_DIR + "/acaf_threshold.chr%d"

CHR = 22
CHR_PFILE = ACAF_PGEN_PATTERN % CHR
for ext in ("pgen", "pvar", "psam"):
    assert os.path.isfile(f"{CHR_PFILE}.{ext}"), f"missing ACAF input: {CHR_PFILE}.{ext}"

# writes go to local scratch, not the gcsfuse-mounted bucket directly -- large
# sequential plink2 output over the network-backed mount is much slower than
# local disk (same lesson as submit_pca_r1.ipynb's ACAF extraction)
LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

QC_PREFIX = os.path.join(LOCAL_WORK_DIR, f"chr{CHR}_round2b_qc")
THIN_PREFIX = os.path.join(LOCAL_WORK_DIR, f"chr{CHR}_round2b_thinned")
QC_NAME = os.path.basename(QC_PREFIX)      # bare filename, for the bucket-copy cell below
THIN_NAME = os.path.basename(THIN_PREFIX)

print(CHR_PFILE)
print(ROUND2B_KEEP_PATH)

## Step 1 — raw variant count (before any QC)

Just `wc -l` on the `.pvar` — no plink2 call, no sample restriction, establishes
the actual starting count for *this* CDR version's chr22 before assuming anything
from public docs. AoU's ACAF threshold callset was ~66.1M variants genome-wide as
of the v8 CDR (population AF > 1% or AC > 100 in any computed ancestry subpop) —
chr22 is ~1.6% of the autosomal genome by length, so a rough prior is on the order
of 1M variants here, but ACAF's per-subpopulation AC/AF criteria mean density isn't
uniform across chromosomes — hence measuring directly rather than trusting the
proportional estimate.

In [ ]:
%%bash -s "$CHR_PFILE"
CHR_PFILE=$1
# ACAF pvar may carry VCF-style meta lines (grep -v '^##'), same as
# submit_pca_r1.ipynb's ACAF handling -- a bare wc -l would overcount
grep -vc '^##' "${CHR_PFILE}.pvar"


## Step 2 — QC (MAF, HWE, missingness, biallelic), restricted to round 2b

One plink2 call: reads ACAF chr22 directly (network-mounted), applies `--keep`
(round 2b passers only — MAF/HWE computed within this cohort, not full AoU) and the
QC filter set, writes to local scratch. Timed with `time` — this is the number that
matters for estimating a genome-wide QC run.

- `--maf 0.01`: min MAF 1%
- `--hwe 1e-6 0.001 keep-fewhet`: HWE 1e-6, `k=0.001` scales the threshold to this
  cohort's sample size (plink2's documented recommendation — see
  `submit_pca_r1.ipynb`'s HWE cell for the same reasoning), `keep-fewhet` only
  excludes *excess*-heterozygosity variants so any residual substructure within
  round 2b's still-not-perfectly-homogeneous cluster isn't mistaken for genotyping
  error
- `--geno 0.05`: missingness < 5%
- `--max-alleles 2` + `--rm-dup exclude-all`: biallelic only, same convention as
  every other QC step in this pipeline
- `--nonfounders`: ACAF's psam has no real pedigree, "founder" status is meaningless
  here — same reasoning as everywhere else in this pipeline

No thinning yet — that's Step 3, applied on top of this QC'd output so re-tuning
the thinning parameter doesn't mean re-running this (expensive) step.

In [ ]:
%%bash -s "$CHR_PFILE" "$ROUND2B_KEEP_PATH" "$QC_PREFIX" "$N_THREADS"
set -e
CHR_PFILE=$1
KEEP_PATH=$2
QC_PREFIX=$3
THREADS=$4

time plink2 \
  --pfile "$CHR_PFILE" \
  --keep "$KEEP_PATH" \
  --maf 0.01 \
  --hwe 1e-6 0.001 keep-fewhet \
  --geno 0.05 \
  --max-alleles 2 \
  --rm-dup exclude-all \
  --nonfounders \
  --threads "$THREADS" \
  --make-pgen \
  --out "$QC_PREFIX"

echo "Post-QC variant count:"
grep -vc '^##' "${QC_PREFIX}.pvar"


## Step 3 — tune thinning toward a ~1M genome-wide target

ACAF's own AF>1%-or-AC>100 filter is a coarse pre-filter, not a tight one — Step 2's
QC almost certainly leaves far more than chr22's ~1.6% share of 1M variants
(~16k). `--thin <p>` (plink2: keep each variant independently with probability `p`)
is the tunable dial: solve for the `p` that would hit the genome-wide target *if*
chr22's post-QC density is representative, then check the actual result.

This is a rough first-pass calibration, not a genome-wide guarantee — chr22 is one
data point, and variant density genuinely varies by chromosome (recombination rate,
gene density). Re-check `p` once a couple more chromosomes are QC'd.

In [ ]:
TARGET_VARIANTS_GENOME = 1_000_000

# chr22 physical length / total autosomal (chr1-22) GRCh38 length -- a proxy for
# chr22's expected share of variants, not a measured one
CHR22_LENGTH_BP = 50_818_468
AUTOSOMAL_LENGTH_BP = 2_875_001_522
CHR22_GENOME_FRACTION = CHR22_LENGTH_BP / AUTOSOMAL_LENGTH_BP

import subprocess
n_chr22_qc = int(subprocess.run(
    ["bash", "-c", f"grep -vc '^##' '{QC_PREFIX}.pvar'"],
    capture_output=True, text=True, check=True
).stdout.strip())

target_chr22 = TARGET_VARIANTS_GENOME * CHR22_GENOME_FRACTION
thin_p = min(1.0, target_chr22 / n_chr22_qc)

print(f"Post-QC chr22 variants: {n_chr22_qc:,}")
print(f"chr22 share of autosomal genome (by length): {CHR22_GENOME_FRACTION:.4f}")
print(f"Target chr22 variants (proportional to {TARGET_VARIANTS_GENOME:,} genome-wide): {target_chr22:,.0f}")
print(f"Thinning probability to try: {thin_p:.4f}")

In [ ]:
%%bash -s "$QC_PREFIX" "$THIN_PREFIX" "$N_THREADS" "$thin_p"
set -e
QC_PREFIX=$1
THIN_PREFIX=$2
THREADS=$3
THIN_P=$4

time plink2 \
  --pfile "$QC_PREFIX" \
  --thin "$THIN_P" \
  --threads "$THREADS" \
  --make-pgen \
  --out "$THIN_PREFIX"

echo "Post-thinning variant count:"
grep -vc '^##' "${THIN_PREFIX}.pvar"


## Persist outputs to the bucket

`LOCAL_WORK_DIR` (`~/scratch_grm`) isn't guaranteed to survive a session
restart -- copy the QC'd and thinned pgen sets (and their `.log`s, which
carry the exact filter counts plink2 reported) to `BUCKET_DIR` so this run
doesn't need repeating just to re-read the numbers. Same lesson as
`submit_pca_r1.ipynb`'s ACAF extraction -- only the small final result
needs to leave local scratch, not the intermediate network-mounted reads.

In [ ]:
%%bash -s "$LOCAL_WORK_DIR" "$BUCKET_DIR" "$QC_NAME" "$THIN_NAME"
set -e
LOCAL_WORK_DIR=$1
BUCKET_DIR=$2
QC_NAME=$3
THIN_NAME=$4

mkdir -p "$BUCKET_DIR"
cp "$LOCAL_WORK_DIR/$QC_NAME".pgen "$LOCAL_WORK_DIR/$QC_NAME".pvar "$LOCAL_WORK_DIR/$QC_NAME".psam "$LOCAL_WORK_DIR/$QC_NAME".log "$BUCKET_DIR/"
cp "$LOCAL_WORK_DIR/$THIN_NAME".pgen "$LOCAL_WORK_DIR/$THIN_NAME".pvar "$LOCAL_WORK_DIR/$THIN_NAME".psam "$LOCAL_WORK_DIR/$THIN_NAME".log "$BUCKET_DIR/"

ls -lh "$BUCKET_DIR"/"$QC_NAME".* "$BUCKET_DIR"/"$THIN_NAME".*

## Summary

`n_chr22_qc` -> `thin_p` -> the actual post-thinning count above: if that count is
close to `target_chr22`, `thin_p` is a reasonable starting point to reuse
genome-wide (scale `TARGET_VARIANTS_GENOME` up/down and re-run Step 3's calculation
cell to retune — no need to redo Step 2's QC). Compare the two `time` outputs
above: Step 2 (QC, reads the full network-mounted chromosome) vs. Step 3 (thinning,
reads only the already-QC'd local file) — Step 2 is the one that matters for
estimating a 22-chromosome genome-wide QC run's total wall-clock time, and for
sizing the shard/batch-job parallelization in the next phase
(`GRM-pairs/grm_bin_sharded`'s `--parallel k n`, not attempted here).